# 05 · Training Loop From Scratch

In plain English, "training" a model just means **showing it examples over and over, measuring how wrong it is, and nudging its numbers in the direction that makes it less wrong**. That repeating cycle is the *training loop*. Every fine-tuning tool you'll ever use — including Hugging Face's famous `Trainer` — is really just this loop with a friendly wrapper around it.

In this notebook we build the whole thing by hand in PyTorch, one line at a time, so that nothing is magic anymore. We'll train a tiny classifier on a few hundred fake 2D points, watch its loss curve slide downward, check its accuracy on data it never trained on, and then *feel* what the famous knobs (learning rate, epochs, batch size) actually do.

## What you'll learn

- How to wrap raw data in a PyTorch **`Dataset`** and feed it in batches with a **`DataLoader`** — and *why* we batch at all.
- The exact meaning of three words people throw around: **epoch**, **batch**, and **iteration**.
- The **canonical 5-line training loop** — `zero_grad → forward → loss → backward → step` — explained line by line, including the classic "why do I need `zero_grad`?" gotcha.
- How to define a small **`nn.Module`** classifier and actually train it.
- How to **collect and plot the loss curve** so you can *see* learning happen.
- How to write a **validation loop** with `model.eval()` and `torch.no_grad()`, and compute **accuracy** on held-out data.
- What the big **hyperparameters** (learning rate, epochs, batch size) do — by changing them and watching the result.
- The difference between **`model.train()`** and **`model.eval()`**, and why it matters.

## Why this matters for fine-tuning

When you fine-tune a large language model, you almost never type out a training loop yourself. You write something like:

```python
trainer = Trainer(model=model, train_dataset=ds, ...)
trainer.train()
```

…and that single `.train()` call runs **the exact loop you're about to build by hand**, just with a giant transformer instead of our tiny classifier.

That matters because the moment something goes wrong — loss is `nan`, accuracy is stuck, training is painfully slow — the error messages and the fixes all live *inside* this loop. The learning rate that "diverges" here is the same learning rate that wrecks a fine-tune. The `zero_grad` we explain here is the line a framework hides but still depends on. **Once you've built the loop once, the `Trainer` stops being a black box and becomes something you can debug.**

## Setup

Run the cell below once. The `%pip install` line is **commented out** — uncomment it if you're on Google Colab or a fresh environment without PyTorch and matplotlib. Everything here runs comfortably on a plain CPU; no GPU needed.

In [ ]:
# Uncomment the next line on Colab or a fresh environment:
# %pip install torch matplotlib

import torch                          # the core deep-learning library (tensors + autograd)
import torch.nn as nn                 # building blocks: layers, loss functions, nn.Module
from torch.utils.data import Dataset, DataLoader  # tools for feeding data in batches
import matplotlib.pyplot as plt       # for plotting the loss curve
import math

torch.manual_seed(0)                  # fix randomness so your numbers match this notebook

print("torch version:", torch.__version__)  # -> e.g. torch version: 2.x.x
print("Imports OK")                          # -> Imports OK

## 1. A tiny dataset we can actually see

Before we can train anything we need data. To keep things visual and CPU-friendly we'll **make up** 200 points in 2D space, split into **two classes (0 and 1)**. Think of it as "two blobs of dots, and the model's job is to tell which blob a dot belongs to."

Each example is a pair: an **input** `x` (two numbers, the coordinates) and a **label** `y` (`0` or `1`). This is the same shape as any classification problem — including text classification, where the two numbers would instead be hundreds of features coming out of a language model.

In [ ]:
# Make two clusters of points. Class 0 sits around (-1, -1); class 1 around (+1, +1).
N = 100  # points PER class -> 200 total

# torch.randn makes random numbers from a bell curve centered at 0.
# We shift one cluster down-left and the other up-right so they're separable-ish.
class0 = torch.randn(N, 2) * 0.7 + torch.tensor([-1.0, -1.0])
class1 = torch.randn(N, 2) * 0.7 + torch.tensor([ 1.0,  1.0])

X = torch.cat([class0, class1], dim=0)            # shape (200, 2): all the inputs
y = torch.cat([torch.zeros(N), torch.ones(N)])    # shape (200,):   all the labels (0s then 1s)
y = y.long()                                       # labels must be integers (long) for the loss

print("X shape:", X.shape)   # -> X shape: torch.Size([200, 2])
print("y shape:", y.shape)   # -> y shape: torch.Size([200])
print("first input:", X[0], "-> label:", y[0].item())

**What this does:** It builds `X`, a `(200, 2)` tensor of coordinates, and `y`, a `(200,)` tensor of `0`/`1` labels. We use `.long()` because PyTorch's classification loss expects integer class ids. A quick plot will show why the model has a fighting chance.

In [ ]:
# Plot the two classes so you can SEE the problem the model must solve.
plt.figure(figsize=(5, 5))
plt.scatter(class0[:, 0], class0[:, 1], label="class 0", alpha=0.6)
plt.scatter(class1[:, 0], class1[:, 1], label="class 1", alpha=0.6)
plt.legend(); plt.title("Our toy dataset"); plt.xlabel("x1"); plt.ylabel("x2")
plt.show()
# Expect: two overlapping-ish blobs, lower-left vs upper-right.

### ✏️ Exercise
Move the clusters **closer together** by changing the `+ torch.tensor([-1.0, -1.0])` and `[1.0, 1.0]` shifts to something like `[-0.3, -0.3]` and `[0.3, 0.3]`. Re-plot. The blobs now overlap heavily — predict whether that will make the model's job *easier* or *harder*, and keep that intuition for later.

## 2. Splitting into training and validation sets

We never judge a model by how well it does on the data it *studied*. A student who memorizes the answer key looks brilliant until the real exam. So we hold out some data the model never sees during training — the **validation set** — and use it as our honest exam.

We'll shuffle the 200 points and keep **160 for training, 40 for validation**.

In [ ]:
# Shuffle the indices 0..199 so classes aren't all grouped together.
perm = torch.randperm(X.shape[0])     # a random ordering of [0, 1, ..., 199]
X, y = X[perm], y[perm]               # reorder both inputs and labels the SAME way

n_train = 160
X_train, y_train = X[:n_train], y[:n_train]   # first 160 -> training
X_val,   y_val   = X[n_train:], y[n_train:]   # last 40  -> validation

print("train:", X_train.shape[0], "examples")  # -> train: 160 examples
print("val:  ", X_val.shape[0],   "examples")  # -> val:   40 examples

**What this does:** `torch.randperm` gives a shuffled list of positions; indexing both `X` and `y` with it keeps each input paired with its correct label. Then we slice off the first 160 for training and the rest for validation. **Always shuffle before splitting**, or you might put all of class 0 in train and all of class 1 in val.

### ✏️ Exercise
Print the **class balance** of your validation set with `y_val.float().mean()`. A value near `0.5` means roughly half are class 1. If it's far from `0.5`, the shuffle gave you a lopsided split — try a different `torch.manual_seed(...)` at the top and re-run.

## 3. `Dataset` and `DataLoader`: feeding data in batches

PyTorch wants two things from you:

1. A **`Dataset`** — an object that knows two methods: `__len__` (how many examples?) and `__getitem__` (give me example number `i`). Think of it as a polite librarian: ask for item 7, get item 7.
2. A **`DataLoader`** — wraps a `Dataset` and hands you data in **batches**, optionally **shuffled** each epoch.

**Why batch at all?** Three reasons:
- **Speed/memory:** processing 32 examples at once is far more efficient than 1-at-a-time, but feeding all 160 (or, for an LLM, millions) at once won't fit in memory.
- **Better gradients:** the average over a small batch is a stable-enough estimate of "which way to nudge the model," without the cost of using the whole dataset.
- **Useful noise:** the slight randomness between batches actually helps models generalize.

In [ ]:
# A Dataset just needs __len__ and __getitem__.
class PointsDataset(Dataset):
    def __init__(self, X, y):
        self.X = X          # store the inputs
        self.y = y          # store the labels

    def __len__(self):
        return len(self.X)  # total number of examples

    def __getitem__(self, idx):
        # Return ONE (input, label) pair. The DataLoader will call this repeatedly.
        return self.X[idx], self.y[idx]

train_ds = PointsDataset(X_train, y_train)
val_ds   = PointsDataset(X_val,   y_val)

print("train_ds length:", len(train_ds))     # -> train_ds length: 160
print("example #0 from dataset:", train_ds[0])  # -> (tensor([...]), tensor(0 or 1))

**What this does:** `PointsDataset` is the librarian. `len(train_ds)` triggers `__len__`; `train_ds[0]` triggers `__getitem__(0)`. That's the whole contract — any object with those two methods is a valid PyTorch dataset, whether it holds 2D points or tokenized sentences.

In [ ]:
# A DataLoader turns the Dataset into an iterable of BATCHES.
BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)   # shuffle each epoch
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)  # no need to shuffle val

# Peek at ONE batch to understand the shapes.
xb, yb = next(iter(train_loader))   # grab the first batch
print("one batch of inputs xb:", xb.shape)   # -> torch.Size([32, 2])  (32 points, 2 coords)
print("one batch of labels yb:", yb.shape)   # -> torch.Size([32])     (32 labels)

# How many batches per epoch? 160 examples / 32 per batch = 5 batches (last may be smaller).
print("batches per epoch:", len(train_loader))  # -> batches per epoch: 5

**What this does:** The `DataLoader` slices the 160 training examples into chunks of 32. Each loop step gives you `xb` (a `(32, 2)` batch of inputs) and `yb` (a `(32,)` batch of labels). `shuffle=True` reshuffles the order every epoch so the model never sees the exact same batch sequence twice — that's good for training.

### ✏️ Exercise
Change `BATCH_SIZE` to `16`, rebuild `train_loader`, and re-print `len(train_loader)`. With 160 examples and batches of 16 you should now get **10** batches per epoch. Notice: smaller batches → more iterations → more weight updates per epoch.

## 4. Epoch, batch, iteration — defined once and for all

These three words confuse everyone at first. Here they are, pinned down:

- **Batch:** a small group of examples processed together (e.g. 32 points). One forward+backward pass uses exactly one batch.
- **Iteration (a.k.a. step):** one batch going through the loop = one weight update. With 5 batches, one pass through the data is 5 iterations.
- **Epoch:** **one full pass over the entire training set.** With 160 examples and batch size 32, one epoch = 5 iterations. Training for 20 epochs = 20 × 5 = 100 iterations total.

A handy mental formula: `iterations_per_epoch = ceil(num_examples / batch_size)`.

In [ ]:
num_examples = len(train_ds)
iters_per_epoch = math.ceil(num_examples / BATCH_SIZE)
print(f"{num_examples} examples / batch {BATCH_SIZE} = {iters_per_epoch} iterations per epoch")
# -> 160 examples / batch 32 = 5 iterations per epoch

EPOCHS = 20
print(f"Training for {EPOCHS} epochs = {EPOCHS * iters_per_epoch} total weight updates")
# -> Training for 20 epochs = 100 total weight updates

**What this does:** Just turns the definitions into arithmetic so the words stick. When a fine-tuning log says "step 4500 / 10000," that's *iterations*, not epochs — now you know the difference.

## 5. Defining the model (a small `nn.Module`)

A PyTorch model is a class that inherits from `nn.Module`. You declare your layers in `__init__`, and describe how data flows through them in `forward`. PyTorch then makes the object **callable**: `model(xb)` automatically runs `forward(xb)` (plus some bookkeeping).

Our classifier is tiny: 2 inputs → a hidden layer of 16 → 2 outputs (one score per class). The `ReLU` in between lets it learn a *curved* boundary instead of just a straight line.

In [ ]:
class TinyClassifier(nn.Module):
    def __init__(self):
        super().__init__()                 # required: set up nn.Module internals
        self.net = nn.Sequential(          # a stack of layers run in order
            nn.Linear(2, 16),              # layer 1: 2 inputs  -> 16 hidden numbers
            nn.ReLU(),                     # non-linearity: keeps positives, zeros negatives
            nn.Linear(16, 2),              # layer 2: 16 hidden -> 2 class scores ("logits")
        )

    def forward(self, x):
        return self.net(x)                 # how an input flows to an output

model = TinyClassifier()
print(model)   # prints the layer structure

# The model is callable: model(xb) runs forward(xb).
example_out = model(xb)                     # xb is our (32, 2) batch from earlier
print("output shape:", example_out.shape)  # -> torch.Size([32, 2])  (2 scores per example)
print("raw scores for first example:", example_out[0])  # untrained -> roughly random numbers

**What this does:** The output is a `(32, 2)` tensor of **logits** — raw, unnormalized scores, one per class. The bigger score wins. Right now the model is untrained (random weights), so its scores are meaningless; training is what makes them informative.

### ✏️ Exercise
Make the model bigger: change the hidden size from `16` to `64` in **both** `nn.Linear` lines (`nn.Linear(2, 64)` and `nn.Linear(64, 2)`). Re-run the cell. More hidden units = more capacity. We'll see whether the toy problem even needs it.

## 6. The loss function and the optimizer

Two more pieces before the loop:

- **Loss function** — a single number measuring *how wrong* the model is on a batch. Lower is better. For classification we use **`nn.CrossEntropyLoss`**, which compares the model's class scores to the true labels. (Bonus: it expects raw logits, so we did *not* add a softmax to the model — it handles that internally.)
- **Optimizer** — the thing that actually *changes the weights*. We give it the model's parameters and a **learning rate** (`lr`), the step size for each nudge. We'll use **Adam**, a popular, forgiving optimizer that's also the default for most LLM fine-tuning.

In [ ]:
loss_fn = nn.CrossEntropyLoss()                          # how-wrong-are-we meter
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)  # the weight-updater

# Sanity check: compute the loss on one batch BEFORE any training.
with torch.no_grad():                       # we're just looking, don't track gradients
    start_loss = loss_fn(model(xb), yb)
print("loss before training:", round(start_loss.item(), 4))
# For 2 classes, a clueless model scores about ln(2) = 0.69. You'll see something near that.

**What this does:** `loss_fn` turns "predictions vs. truth" into one number. `optimizer` holds `model.parameters()` (all the learnable weights) and the learning rate. The starting loss near `0.69` (= `ln(2)`) is the signature of a model guessing at random between two classes — a useful baseline to confirm before we train.

## 7. The canonical training loop, line by line

Here is the heart of everything. **Memorize this shape** — it is the same five steps in every PyTorch project, from this toy to a billion-parameter fine-tune:

```python
optimizer.zero_grad()    # 1. reset gradients from the previous step
preds = model(xb)        # 2. forward pass: get predictions
loss  = loss_fn(preds, yb)  # 3. measure how wrong we are
loss.backward()          # 4. backward pass: compute gradients
optimizer.step()         # 5. update the weights using those gradients
```

**Why `optimizer.zero_grad()`?** This trips up nearly everyone. In PyTorch, calling `loss.backward()` *adds* the new gradients on top of whatever is already stored — it **accumulates** rather than replaces. If you forget to zero them, batch 2's update is polluted by batch 1's leftover gradients, and training quietly goes haywire. So we wipe the slate clean at the start of every step.

We also call **`model.train()`** before the loop. It switches the model into "training mode" (relevant for layers like dropout and batch-norm). Our tiny model has none of those, but it's a habit worth building now because real models do.

In [ ]:
model.train()              # put model in TRAINING mode (matters for dropout/batchnorm later)
train_losses = []          # we'll record the average loss each epoch to plot it

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for xb, yb in train_loader:        # one pass over all batches = one epoch
        optimizer.zero_grad()          # 1. clear old gradients (they accumulate!)
        preds = model(xb)              # 2. forward pass: predictions for this batch
        loss = loss_fn(preds, yb)      # 3. how wrong are we on this batch?
        loss.backward()                # 4. compute gradients via backpropagation
        optimizer.step()               # 5. nudge every weight a little to reduce the loss
        epoch_loss += loss.item()      # accumulate the (plain Python) loss number

    avg_loss = epoch_loss / len(train_loader)   # average loss over the epoch's batches
    train_losses.append(avg_loss)
    if epoch % 2 == 0 or epoch == EPOCHS - 1:   # print every other epoch to stay tidy
        print(f"epoch {epoch:2d} | train loss {avg_loss:.4f}")

print("Done training.")
# Expect the printed loss to fall steadily, e.g. ~0.69 down toward ~0.10-0.20.

**What this does:** The outer loop counts epochs; the inner loop walks through batches. Each batch runs the five sacred lines. `loss.item()` pulls a plain Python float out of the tensor so we can average and store it. You should watch the printed loss **decrease** epoch over epoch — that *is* learning, happening in front of you.

### ✏️ Exercise
**Comment out the `optimizer.zero_grad()` line** and re-run this cell (re-create the model first by re-running Section 5 and 6 so you start fresh). Watch what happens to the loss. With gradients piling up uncontrolled, training usually destabilizes or stalls. Then put the line back. This is the single most common beginner bug — now you've seen it.

## 8. Plotting the loss curve

Numbers in a log are fine, but a **picture** tells you instantly whether training is healthy. A good loss curve drops quickly, then flattens (the model has learned most of what it can). A curve that's flat from the start means nothing is being learned; one that's spiky or rising means your learning rate is probably too high.

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(range(1, EPOCHS + 1), train_losses, marker="o")
plt.xlabel("epoch"); plt.ylabel("training loss")
plt.title("Loss curve (should go DOWN)")
plt.grid(True, alpha=0.3)
plt.show()
# Expect: a line starting near 0.69 and sloping downward, flattening near the end.

**What this does:** Plots the per-epoch average loss we collected. The shape — steep then flat — is the classic "learning is working" signature. Reading loss curves is a core fine-tuning skill; the `Trainer` logs the very same numbers.

## 9. The validation loop: an honest exam

Training loss tells you how well the model fits data it *studied*. **Validation** tells you how well it does on data it has **never seen** — the number that actually matters. Two new tools appear here:

- **`model.eval()`** — switches the model to evaluation mode (turns off dropout/batchnorm randomness). Pair it with `model.train()` when you go back to training.
- **`torch.no_grad()`** — tells PyTorch "don't track gradients, we're not learning right now." This makes evaluation faster and uses less memory, since we never call `backward()` here.

We'll compute **accuracy**: the fraction of validation points the model labels correctly.

In [ ]:
def evaluate(model, loader):
    model.eval()                       # EVALUATION mode (no dropout/batchnorm noise)
    correct, total = 0, 0
    with torch.no_grad():              # don't build the gradient graph; just predict
        for xb, yb in loader:
            preds = model(xb)          # (batch, 2) logits
            predicted = preds.argmax(dim=1)   # pick the class with the higher score
            correct += (predicted == yb).sum().item()   # count the hits
            total += yb.shape[0]
    return correct / total             # accuracy = right / total

val_acc = evaluate(model, val_loader)
print(f"validation accuracy: {val_acc:.2%}")
# Expect something high, e.g. ~90-100%, since the two blobs are fairly separable.

train_acc = evaluate(model, train_loader)
print(f"training accuracy:   {train_acc:.2%}")
# Train accuracy is usually a touch higher than validation accuracy.

**What this does:** `argmax(dim=1)` turns the two per-class scores into a single predicted label (whichever score is bigger). We compare predictions to truth, count correct ones, and divide. Because we wrapped it in `model.eval()` + `torch.no_grad()`, this is a clean, fast, gradient-free measurement. **A big gap** where training accuracy is much higher than validation accuracy is the warning sign of *overfitting*.

### ✏️ Exercise
After calling `evaluate` (which sets `model.eval()`), the model is left in eval mode. Add a line `model.train()` right after evaluating if you intend to keep training. Then call `evaluate` again but on `train_loader` and confirm you can reproduce the printed training accuracy.

## 10. Feeling the hyperparameters

A **hyperparameter** is a setting *you* choose before training (as opposed to the weights, which the model learns). The three you'll tune constantly — in this toy and in every fine-tune — are:

- **Learning rate (`lr`)** — the step size of each weight nudge. **Too high** and the loss bounces around or explodes to `nan` (you overshoot the target every time). **Too low** and training crawls (tiny steps take forever).
- **Epochs** — how many full passes over the data. Too few = underfit (didn't learn enough); too many = wasted time and risk of overfitting.
- **Batch size** — how many examples per step. Smaller = more updates per epoch and noisier gradients; larger = smoother but fewer updates.

Let's package training into a function so we can twist these knobs and *watch* the effect.

In [ ]:
def train_model(lr=0.05, epochs=20, batch_size=32):
    """Train a fresh TinyClassifier and return its per-epoch loss curve + final val accuracy."""
    torch.manual_seed(0)                       # same starting weights every time = fair comparison
    m = TinyClassifier()
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    losses = []
    m.train()
    for _ in range(epochs):
        ep = 0.0
        for xb, yb in loader:
            opt.zero_grad()
            loss = loss_fn(m(xb), yb)
            loss.backward()
            opt.step()
            ep += loss.item()
        losses.append(ep / len(loader))

    acc = evaluate(m, val_loader)
    return losses, acc

# Baseline run:
base_losses, base_acc = train_model(lr=0.05, epochs=20, batch_size=32)
print(f"baseline final loss {base_losses[-1]:.4f} | val acc {base_acc:.2%}")

**What this does:** Wraps the whole loop in one reusable function. Crucially it re-seeds and rebuilds the model each call, so any difference we see comes from the *hyperparameter we changed*, not luck. Now the experiments below are apples-to-apples.

In [ ]:
# Compare three learning rates on the SAME problem.
plt.figure(figsize=(7, 4))
for lr in [0.001, 0.05, 2.0]:          # too low, just right, way too high
    losses, acc = train_model(lr=lr, epochs=20, batch_size=32)
    plt.plot(range(1, 21), losses, marker=".", label=f"lr={lr} (val acc {acc:.0%})")
plt.xlabel("epoch"); plt.ylabel("train loss"); plt.legend()
plt.title("Learning rate: too low vs. good vs. too high")
plt.grid(True, alpha=0.3)
plt.show()

# Expect:
#   lr=0.001 -> loss barely moves (too slow)
#   lr=0.05  -> loss drops nicely (good)
#   lr=2.0   -> loss is erratic / high / may not improve (too high, overshooting)

**What this does:** Runs the same training three times, changing only the learning rate, and overlays the loss curves. This single picture teaches the most important hyperparameter lesson in deep learning: there's a "Goldilocks" learning rate, and both extremes hurt. The exact same trade-off governs LLM fine-tuning — just with much smaller numbers like `2e-5`.

### ✏️ Exercise
Run your own sweep over **batch size**: call `train_model(batch_size=b)` for `b` in `[8, 32, 160]` and print each final loss and val accuracy. `batch_size=160` means the whole training set is one batch (so only 1 update per epoch) — notice it needs more epochs to reach the same loss. Then sweep **epochs** over `[5, 20, 100]` and watch where the loss stops improving.

## Common mistakes & how to debug them

- **Forgot `optimizer.zero_grad()`** → gradients accumulate across batches; loss behaves erratically or won't go down. **Fix:** make it the *first* line of the loop.
- **Loss is `nan` or explodes** → almost always a **learning rate that's too high**. **Fix:** drop `lr` by 10× (e.g. `0.05 → 0.005`) and re-run.
- **Loss never moves / accuracy stuck at ~50%** → learning rate too low, too few epochs, or you forgot `loss.backward()` / `optimizer.step()`. **Fix:** confirm all five loop lines are present and try a bigger `lr`.
- **Labels wrong dtype** → `CrossEntropyLoss` needs integer (`long`) labels and **raw logits** (no softmax in the model). A cryptic dtype/shape error here usually means one of those. **Fix:** `y = y.long()`, and don't add a final `softmax` layer.
- **Forgot `model.eval()` / `torch.no_grad()` during validation** → metrics can be subtly off (if you have dropout/batchnorm) and evaluation is slower and memory-hungry. **Fix:** always wrap evaluation in both.
- **Left the model in `.eval()` mode and kept training** → dropout/batchnorm won't behave correctly. **Fix:** call `model.train()` before resuming the loop.
- **Evaluating on training data and calling it "accuracy"** → flatteringly high but meaningless. **Fix:** always report the **validation** number.

## Summary

You built a complete training loop from scratch and now understand every moving part:

- A **`Dataset`** answers "how many?" and "give me item `i`"; a **`DataLoader`** turns it into shuffled **batches**.
- **Epoch** = one full pass over the data; **iteration/step** = one batch = one weight update; `iterations_per_epoch = ceil(N / batch_size)`.
- The **canonical loop** is five lines: `zero_grad → forward → loss → backward → step`, and `zero_grad` is non-negotiable because gradients **accumulate**.
- You watched a **loss curve** slide downward — the visual signature of learning.
- A **validation loop** with `model.eval()` + `torch.no_grad()` gives an honest **accuracy** on unseen data.
- **Learning rate, epochs, and batch size** are the knobs you'll tune forever; you saw a too-high `lr` misbehave and a too-low one crawl.

Most importantly: **Hugging Face's `Trainer` runs exactly this loop for you.** When you call `trainer.train()` in the upcoming notebooks, you now know what's happening under the hood — and where to look when it breaks.

## What to learn next

Next up: **`06_transformers_and_attention.ipynb`**. We've trained a tiny classifier on 2D points; now we'll meet the architecture that powers every modern LLM — the **transformer** — and the **attention** mechanism at its core. You'll see how the same training loop you just mastered scales up to models that understand language, setting the stage for actually fine-tuning one.